# YOLO26 Solar Filament Crop Preprocessing

Stage 1 of the pipeline. This notebook audits annotations, creates the filename-grouped split, generates and audits YOLO crops, and writes a reusable handoff bundle. It does not import PyTorch or load a model.

In Kaggle, set Accelerator to None before running this notebook. Crop generation uses four CPU processes by default (about 400% CPU when Kaggle grants four cores). Set FILAMENT_CROP_WORKERS to override the worker count.

Set DEBUG=True for per-image diagnostic logs. Set it to False to keep only stage-level INFO logs.


## 1. CPU preprocessing configuration


In [ ]:
from pathlib import Path
import os,logging
DEBUG=True
logging.basicConfig(level=logging.DEBUG if DEBUG else logging.INFO,format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',force=True)
LOGGER=logging.getLogger('filament_preprocessing')
LOGGER.info('DEBUG=%s; preprocessing log level=%s',DEBUG,logging.getLevelName(logging.getLogger().level))
CPU_CROP_WORKERS=int(os.environ.get('FILAMENT_CROP_WORKERS','4'))
SEED=42
VALIDATION_FRACTION=.20
CROP_SIZE=640
GRID_STRIDE=480
TARGET_CROP_STRIDE=480
MIN_VISIBLE_MASK_AREA=8
MIN_COMPONENT_AREA=1
POLYGON_SIMPLIFY_EPSILON=.25
EDGE_PAD_VALUE=0
NEGATIVE_SAFETY_MARGIN=16
NEGATIVE_FRACTION_TARGET=.25
MAX_NEGATIVE_CROPS_PER_VIEW=8
MAX_GRID_CROPS_PER_VIEW=8
MAX_TARGET_CROPS_PER_ANNOTATION=32
DATA_ROOT=None
REBUILD_DATASET=False
STRICT_POLYGON_CONVERSION=True
WORKING_ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
ARTIFACT_DIR=WORKING_ROOT/'yolo26_crop_training_artifacts'
YOLO_DATA_DIR=ARTIFACT_DIR/'prepared_dataset'
if CPU_CROP_WORKERS<1:raise ValueError('FILAMENT_CROP_WORKERS must be at least 1.')
print(f'Preprocessing output: {ARTIFACT_DIR} | crop workers: {CPU_CROP_WORKERS}')


## 2. CPU-only imports and environment setup


In [ ]:
import concurrent.futures
import hashlib,json,math,os,platform,random,shutil,sys,time
from collections import Counter,defaultdict
from datetime import datetime,timezone
import multiprocessing as mp
try:
    import cv2,matplotlib.pyplot as plt,numpy as np,pandas as pd
    from IPython.display import display
    from pycocotools.coco import COCO
    from sklearn.model_selection import GroupShuffleSplit
    from threadpoolctl import threadpool_limits
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError(f'Missing preprocessing dependency {exc}.') from exc
random.seed(SEED);np.random.seed(SEED)
cv2.setNumThreads(1)
CPU_AFFINITY_COUNT=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else (os.cpu_count() or 1)
CPU_AFFINITY_COUNT=max(1,CPU_AFFINITY_COUNT)
THREADPOOL_LIMITER=threadpool_limits(limits=1)
LOGGER.info('Preprocessing runtime ready | Python %s | OpenCV %s',platform.python_version(),cv2.__version__)
LOGGER.info('CPU affinity: %s cores | crop processes: %s | native threads per process: 1',CPU_AFFINITY_COUNT,CPU_CROP_WORKERS)
LOGGER.debug('CPU inventory: os.cpu_count=%s; affinity=%s; multiprocessing start methods=%s',os.cpu_count(),CPU_AFFINITY_COUNT,mp.get_all_start_methods())


## 4. Dataset discovery

In [ ]:
ANNOTATION_FILENAME='MAGFiLO_1.0_Annotations_kaggle2026_train.json'
def find_dataset_root(explicit=None):
    roots=[Path(explicit).expanduser()] if explicit else [Path.cwd(),Path.cwd().parent,Path('/kaggle/input')]
    for parent in roots:
        if not parent.exists(): continue
        candidates=[parent/'train'/ANNOTATION_FILENAME]
        candidates=[p for p in candidates if p.is_file()]
        if not candidates:
            try: candidates=sorted(parent.glob(f'**/{ANNOTATION_FILENAME}'))
            except OSError: candidates=[]
        for annotation in candidates:
            root=annotation.parent.parent
            if (root/'train'/'train_images').is_dir() and (root/'test'/'test_images').is_dir():
                return root.resolve()
    raise FileNotFoundError('Could not find the MAGFiLO dataset; set DATA_ROOT explicitly.')
DATA_ROOT=find_dataset_root(DATA_ROOT)
TRAIN_IMAGE_DIR=DATA_ROOT/'train'/'train_images'
TEST_IMAGE_DIR=DATA_ROOT/'test'/'test_images'
ANNOTATION_PATH=DATA_ROOT/'train'/ANNOTATION_FILENAME
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
LOGGER.info('Dataset: %s | test files: %s',DATA_ROOT,sum(p.is_file() for p in TEST_IMAGE_DIR.iterdir()))
LOGGER.debug('Input paths: train_images=%s annotations=%s output=%s',TRAIN_IMAGE_DIR,ANNOTATION_PATH,ARTIFACT_DIR)

## 4. COCO annotation and annotation-view audit


In [ ]:
coco_json=json.loads(ANNOTATION_PATH.read_text(encoding='utf-8'))
coco=COCO(str(ANNOTATION_PATH))
raw_images=coco_json.get('images',[]);raw_annotations=coco_json.get('annotations',[])
if not raw_images or not raw_annotations: raise ValueError('Empty COCO images or annotations.')
image_rows=[];image_by_id={}
for record in raw_images:  # Preserve JSON order for split parity.
    iid,name=str(record['id']),str(record['file_name'])
    if iid in image_by_id: raise ValueError(f'Duplicate image_id: {iid}')
    image_by_id[iid]=record
    if '-' not in iid or iid.split('-',1)[1]!=Path(name).stem:
        raise ValueError(f'Unverified annotation-view mapping: {iid} -> {name}')
    image_rows.append({'image_id':iid,'file_name':name,'physical_filename':name,
        'annotation_view_id':iid.split('-',1)[0],'width':int(record['width']),'height':int(record['height'])})
records=pd.DataFrame(image_rows)
annotation_ids_by_image=defaultdict(list);seen_ann=set()
for ann in raw_annotations:
    iid,aid=str(ann['image_id']),str(ann['id'])
    if iid not in image_by_id: raise ValueError(f'Annotation {aid} points to unknown image {iid}')
    if aid in seen_ann: raise ValueError(f'Duplicate annotation id {aid}')
    seen_ann.add(aid);annotation_ids_by_image[iid].append(aid)
records['annotation_count']=records.image_id.map(lambda iid:len(annotation_ids_by_image[iid]))
if (records.annotation_count==0).any(): raise ValueError('COCO view without annotations.')
missing=[n for n in records.file_name.unique() if not (TRAIN_IMAGE_DIR/n).is_file()]
if missing: raise FileNotFoundError(f'Missing images: {missing[:10]}')
view_manifest=records[['image_id','file_name','annotation_view_id','annotation_count','width','height']].copy()
view_manifest['annotation_ids']=view_manifest.image_id.map(lambda iid:';'.join(annotation_ids_by_image[iid]))
view_manifest.to_csv(ARTIFACT_DIR/'annotation_view_mapping.csv',index=False)
LOGGER.info('COCO audit passed | physical images=%s | views=%s | instances=%s',records.file_name.nunique(),len(records),len(raw_annotations))
LOGGER.debug('Views per physical image: %s',records.groupby('file_name').image_id.nunique().value_counts().sort_index().to_dict())
LOGGER.debug('Categories mapped to filament: %s',dict(Counter(a['category_id'] for a in raw_annotations)))
display(records.head())

## 5. Filename-grouped train/validation split


In [ ]:
splitter=GroupShuffleSplit(n_splits=1,test_size=VALIDATION_FRACTION,random_state=SEED)
train_idx,val_idx=next(splitter.split(records,groups=records.file_name))
split_manifest=records[['file_name','image_id','annotation_view_id','annotation_count']].copy()
split_manifest['split']='val';split_manifest.loc[train_idx,'split']='train'
if split_manifest.groupby('file_name').split.nunique().max()!=1:
    raise AssertionError('Annotation views of a physical filename cross splits.')
train_files=set(split_manifest.loc[split_manifest.split=='train','file_name'])
val_files=set(split_manifest.loc[split_manifest.split=='val','file_name'])
if train_files&val_files: raise AssertionError('Physical-image leakage.')
split_manifest_path=ARTIFACT_DIR/'split_manifest.csv'
split_manifest.to_csv(split_manifest_path,index=False)
YOLO_DATA_DIR.mkdir(parents=True,exist_ok=True)
split_manifest.to_csv(YOLO_DATA_DIR/'split_manifest.csv',index=False)
split_identifier=hashlib.sha256(split_manifest[['file_name','split']].drop_duplicates()
    .sort_values('file_name').to_csv(index=False).encode()).hexdigest()[:16]
print('Train physical/views:',len(train_files),sum(split_manifest.split=='train'))
print('Validation physical/views:',len(val_files),sum(split_manifest.split=='val'))
print('Split ID:',split_identifier)

## 6. Crop and polygon utilities


In [ ]:
def crop_origins(lo,hi,size=CROP_SIZE,stride=TARGET_CROP_STRIDE):
    lo,hi=int(math.floor(lo)),int(math.ceil(hi))
    if hi-lo<=size:return [int(math.floor((lo+hi-size)/2))]
    last=hi-size;out=list(range(lo,last+1,stride))
    if not out or out[-1]!=last:out.append(last)
    return sorted(set(out))
def grid_origins(length,size=CROP_SIZE,stride=GRID_STRIDE):
    if length<=size:return [int(math.floor((length-size)/2))]
    last=length-size;out=list(range(0,last+1,stride))
    if out[-1]!=last:out.append(last)
    return out
def crop_pad(a,x,y,size=CROP_SIZE,pad=0):
    h,w=a.shape[:2];shape=(size,size)+(() if a.ndim==2 else (a.shape[2],))
    out=np.full(shape,pad,dtype=a.dtype)
    sx0,sy0,sx1,sy1=max(0,x),max(0,y),min(w,x+size),min(h,y+size)
    if sx1>sx0 and sy1>sy0:
        dx,dy=sx0-x,sy0-y
        out[dy:dy+sy1-sy0,dx:dx+sx1-sx0]=a[sy0:sy1,sx0:sx1]
    return out
def mask_bbox(mask):
    ys,xs=np.where(mask)
    return None if not len(xs) else (int(xs.min()),int(ys.min()),int(xs.max())+1,int(ys.max())+1)
def polygon_segments(mask):
    binary=(np.asarray(mask)>0).astype(np.uint8);h,w=binary.shape
    n,labels,stats,_=cv2.connectedComponentsWithStats(binary,connectivity=8)
    _,hierarchy=cv2.findContours(binary,cv2.RETR_CCOMP,cv2.CHAIN_APPROX_SIMPLE)
    holes=int((hierarchy[0][:,3]>=0).sum()) if hierarchy is not None else 0
    polys=[];diag={'small_components':0,'invalid_components':0,'component_count':n-1,'hole_count':holes}
    for k in range(1,n):
        if int(stats[k,cv2.CC_STAT_AREA])<MIN_COMPONENT_AREA:
            diag['small_components']+=1;continue
        component=(labels==k).astype(np.uint8)
        contours,_=cv2.findContours(component,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE)
        if not contours:diag['invalid_components']+=1;continue
        contour=max(contours,key=cv2.contourArea)
        if POLYGON_SIMPLIFY_EPSILON>0:
            contour=cv2.approxPolyDP(contour,POLYGON_SIMPLIFY_EPSILON,True)
        points=contour.reshape(-1,2).astype(np.float32)
        if len(points)<3 or cv2.contourArea(contour)<=0:
            x,y,bw,bh=cv2.boundingRect(component)
            points=np.asarray([[x-.5,y-.5],[x+bw-.5,y-.5],[x+bw-.5,y+bh-.5],[x-.5,y+bh-.5]],np.float32)
        points[:,0]=np.clip(points[:,0]/w,0,1);points[:,1]=np.clip(points[:,1]/h,0,1)
        if len(np.unique(points,axis=0))<3:diag['invalid_components']+=1
        else:polys.append(points)
    return polys,diag
def polygons_mask(polys):
    out=np.zeros((CROP_SIZE,CROP_SIZE),np.uint8)
    for p in polys:
        xy=np.rint(p*np.array([CROP_SIZE,CROP_SIZE])).astype(np.int32)
        xy[:,0]=np.clip(xy[:,0],0,CROP_SIZE-1);xy[:,1]=np.clip(xy[:,1],0,CROP_SIZE-1)
        cv2.fillPoly(out,[xy],1)
    return out.astype(bool)
def load_masks(iid):
    out=[]
    for ann in coco.loadAnns(coco.getAnnIds(imgIds=[iid])):
        if str(ann['image_id'])!=str(iid):raise ValueError(f'Wrong annotation mapping: {ann["id"]}')
        mask=coco.annToMask(ann).astype(bool)
        out.append({'annotation_id':str(ann['id']),'mask':mask,'bbox':mask_bbox(mask)})
    return out
def visible(masks,x,y):
    out=[]
    for item in masks:
        m=crop_pad(item['mask'].astype(np.uint8),x,y).astype(bool);area=int(m.sum())
        if area>=MIN_VISIBLE_MASK_AREA:out.append((item,m,area))
    return out
def subsample(items,maximum):
    items=sorted(set(items))
    if len(items)<=maximum:return items
    indexes=np.linspace(0,len(items)-1,num=maximum,dtype=int)
    return [items[i] for i in sorted(set(indexes.tolist()))]
_test=np.zeros((CROP_SIZE,CROP_SIZE),np.uint8);_test[100:120,110:140]=1
_p,_d=polygon_segments(_test)
assert _p and polygons_mask(_p).any()
assert crop_pad(np.ones((4,4),np.uint8),-1,-1,4).shape==(4,4)
print('Padding and polygon sanity checks passed.')

## 7. Crop I/O, background safety, and caching


In [ ]:
def safety_mask(image_ids,shape):
    known=np.zeros(shape,np.uint8)
    for iid in image_ids:
        for item in load_masks(iid):known|=item['mask'].astype(np.uint8)
    if NEGATIVE_SAFETY_MARGIN and known.any():
        d=2*NEGATIVE_SAFETY_MARGIN+1
        known=cv2.dilate(known,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(d,d)))
    return known.astype(bool)
def crop_id(iid,x,y,strategy):
    return hashlib.sha1(f'{iid}|{x}|{y}|{strategy}'.encode()).hexdigest()[:16]
def write_crop(split,name,iid,view,image,masks,x,y,strategy,targets=(),safe=None):
    shown=visible(masks,x,y);shown_ids={a['annotation_id'] for a,_,_ in shown}
    targets=sorted(set(map(str,targets)))
    if not set(targets)<=shown_ids:return None
    if strategy=='negative':
        if safe is None or crop_pad(safe.astype(np.uint8),x,y).any():return None
        shown=[]
    cid=crop_id(iid,x,y,strategy);imrel=Path('images')/split/f'{cid}.jpg';labrel=Path('labels')/split/f'{cid}.txt'
    impath,lpath=YOLO_DATA_DIR/imrel,YOLO_DATA_DIR/labrel
    impath.parent.mkdir(parents=True,exist_ok=True);lpath.parent.mkdir(parents=True,exist_ok=True)
    if not cv2.imwrite(str(impath),crop_pad(image,x,y,pad=EDGE_PAD_VALUE),[cv2.IMWRITE_JPEG_QUALITY,95]):
        raise OSError(f'Could not write {impath}')
    lines=[];ids=[];areas=[];iou_info=[];conversion=Counter()
    for item,mask,area in shown:
        polys,diag=polygon_segments(mask)
        conversion.update(diag)
        if diag['invalid_components'] and STRICT_POLYGON_CONVERSION:
            impath.unlink(missing_ok=True);raise ValueError(f'Invalid polygon {item["annotation_id"]} in {cid}')
        if polys:
            ids.append(item['annotation_id']);areas.append(f'{item["annotation_id"]}:{area}')
            lines.extend('0 '+' '.join(f'{v:.6f}' for v in p.reshape(-1)) for p in polys)
            iou_info.append((item['annotation_id'],mask,polys))
    if not set(targets)<=set(ids):impath.unlink(missing_ok=True);raise ValueError(f'Target label lost in {cid}')
    lpath.write_text('\n'.join(lines)+('\n' if lines else ''),encoding='utf-8')
    return {'crop_id':cid,'split':split,'physical_filename':name,'file_name':name,
        'image_id':iid,'annotation_view_id':view,'source_annotation_ids':';'.join(ids),
        'target_annotation_ids':';'.join(targets),'crop_x0':x,'crop_y0':y,
        'crop_x1':x+CROP_SIZE,'crop_y1':y+CROP_SIZE,'crop_size':CROP_SIZE,
        'strategy':strategy,'image_path':imrel.as_posix(),'label_path':labrel.as_posix(),
        'instance_rows':len(lines),'visible_mask_area':sum(a for _,_,a in shown),
        'visible_annotation_areas':';'.join(areas),'negative_safe_checked':strategy=='negative',
        'polygon_components':int(conversion['component_count']),'polygon_holes':int(conversion['hole_count']),
        'polygon_small_components_dropped':int(conversion['small_components']),
        'polygon_invalid_components':int(conversion['invalid_components'])},iou_info
def source_signature():
    files=[]
    for name in sorted(records.file_name.unique()):
        st=(TRAIN_IMAGE_DIR/name).stat();files.append((name,st.st_size,st.st_mtime_ns))
    st=ANNOTATION_PATH.stat()
    keys=['SEED','VALIDATION_FRACTION','CROP_SIZE','GRID_STRIDE','TARGET_CROP_STRIDE','MIN_VISIBLE_MASK_AREA',
      'MIN_COMPONENT_AREA','POLYGON_SIMPLIFY_EPSILON','EDGE_PAD_VALUE','NEGATIVE_SAFETY_MARGIN',
      'NEGATIVE_FRACTION_TARGET','MAX_NEGATIVE_CROPS_PER_VIEW','MAX_GRID_CROPS_PER_VIEW',
      'MAX_TARGET_CROPS_PER_ANNOTATION']
    payload={'annotation_size':st.st_size,'annotation_mtime_ns':st.st_mtime_ns,'files':files,
        'split_identifier':split_identifier,'config':{k:globals()[k] for k in keys}}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
def cache_crop_dataset(signature):
    mp,ap=YOLO_DATA_DIR/'crop_manifest.csv',YOLO_DATA_DIR/'dataset_audit.json'
    if REBUILD_DATASET or not mp.is_file() or not ap.is_file():return None
    try:
        audit=json.loads(ap.read_text(encoding='utf-8'))
        if audit.get('source_preprocessing_signature')!=signature:return None
        m=pd.read_csv(mp)
        if m.crop_id.duplicated().any():return None
        if all((YOLO_DATA_DIR/p).is_file() for p in m.image_path) and all((YOLO_DATA_DIR/p).is_file() for p in m.label_path):return m
    except Exception:return None
    return None
def clear_generated_dataset():
    dest,src=YOLO_DATA_DIR.resolve(),DATA_ROOT.resolve()
    if dest==src or src in dest.parents:raise RuntimeError('Refusing to clear output inside source dataset.')
    for folder in ['images','labels']:
        p=YOLO_DATA_DIR/folder
        if p.exists():shutil.rmtree(p)
    for filename in ['crop_manifest.csv','dataset_audit.json','dataset.yaml','polygon_iou_sample.json']:
        (YOLO_DATA_DIR/filename).unlink(missing_ok=True)



## 8. Parallel crop generation and label audit


In [ ]:
def init_crop_worker():
    global _CROP_THREADPOOL_LIMITER
    cv2.setNumThreads(1)
    _CROP_THREADPOOL_LIMITER=threadpool_limits(limits=1)

def process_physical_image(name):
    group=RECORDS_BY_FILE[name]
    LOGGER.debug('Cropping source image %s (%s annotation views)',name,len(group))
    image=cv2.imread(str(TRAIN_IMAGE_DIR/name),cv2.IMREAD_GRAYSCALE)
    if image is None:raise OSError(f'Unreadable source: {name}')
    shapes={(int(r.height),int(r.width)) for r in group.itertuples(index=False)}
    if shapes!={image.shape}:raise ValueError(f'Dimension mismatch for {name}: {image.shape}')
    iids=group.image_id.astype(str).tolist()
    safe=safety_mask(iids,image.shape)
    splits=set(group.image_id.map(SPLIT_OF_IMAGE))
    if len(splits)!=1:raise AssertionError(f'Views cross split for {name}')
    split=splits.pop()
    rows=[];iou_samples=[]
    for rec in group.itertuples(index=False):
        iid,view=str(rec.image_id),str(rec.annotation_view_id)
        masks=load_masks(iid);centered=defaultdict(set)
        for item in masks:
            if item['bbox'] is None:continue
            x0,y0,x1,y1=item['bbox']
            for y in crop_origins(y0,y1):
                for x in crop_origins(x0,x1):centered[(x,y)].add(item['annotation_id'])
        grid=[(x,y) for y in grid_origins(image.shape[0]) for x in grid_origins(image.shape[1])
              if (x,y) not in centered and visible(masks,x,y)]
        grid=subsample(grid,MAX_GRID_CROPS_PER_VIEW);begin=len(rows)
        for (x,y),targets in sorted(centered.items(),key=lambda z:(z[0][1],z[0][0])):
            item=write_crop(split,name,iid,view,image,masks,x,y,'filament_centered',targets)
            if item:
                row,iou=item;rows.append(row)
                for aid,truth,polys in iou:
                    if len(iou_samples)<64:
                        pred=polygons_mask(polys);union=int((pred|truth).sum())
                        iou_samples.append({'crop_id':row['crop_id'],'annotation_id':aid,
                            'mask_reconstruction_iou':int((pred&truth).sum())/union if union else 1.0})
        for x,y in grid:
            item=write_crop(split,name,iid,view,image,masks,x,y,'grid_context')
            if item:rows.append(item[0])
        npos=len(rows)-begin
        nneg=min(MAX_NEGATIVE_CROPS_PER_VIEW,math.ceil(NEGATIVE_FRACTION_TARGET/(1-NEGATIVE_FRACTION_TARGET)*npos))
        if nneg:
            candidates=[(x,y) for y in grid_origins(image.shape[0]) for x in grid_origins(image.shape[1])
                        if not crop_pad(safe.astype(np.uint8),x,y).any()]
            seedval=int(hashlib.sha256(iid.encode()).hexdigest()[:8],16);rng=random.Random(SEED+seedval);rng.shuffle(candidates)
            attempts=0
            while len(candidates)<nneg and attempts<100:
                attempts+=1;x=rng.randint(-CROP_SIZE//2,image.shape[1]-CROP_SIZE//2)
                y=rng.randint(-CROP_SIZE//2,image.shape[0]-CROP_SIZE//2)
                if not crop_pad(safe.astype(np.uint8),x,y).any() and (x,y) not in candidates:candidates.append((x,y))
            for x,y in candidates[:nneg]:
                item=write_crop(split,name,iid,view,image,[],x,y,'negative',safe=safe)
                if item:rows.append(item[0])
    del image,safe
    return rows,iou_samples

def generate_crops():
    signature=source_signature();cached=cache_crop_dataset(signature)
    if cached is not None:
        LOGGER.info('Reusing matching crop cache with %s rows.',len(cached));return cached,signature
    clear_generated_dataset()
    global RECORDS_BY_FILE,SPLIT_OF_IMAGE
    RECORDS_BY_FILE={n:g for n,g in records.groupby('file_name',sort=False)}
    SPLIT_OF_IMAGE=split_manifest.set_index('image_id').split.to_dict()
    names=sorted(RECORDS_BY_FILE);rows=[];iou_samples=[]
    use_processes=CPU_CROP_WORKERS>1 and 'fork' in mp.get_all_start_methods()
    if use_processes:
        LOGGER.info('Generating crops with %s CPU processes (one native thread each).',CPU_CROP_WORKERS)
        with concurrent.futures.ProcessPoolExecutor(
            max_workers=CPU_CROP_WORKERS,mp_context=mp.get_context('fork'),
            initializer=init_crop_worker
        ) as executor:
            completed=executor.map(process_physical_image,names,chunksize=1)
            for image_rows,image_iou in tqdm(completed,total=len(names),desc='Generate crops'):
                rows.extend(image_rows)
                if len(iou_samples)<64:iou_samples.extend(image_iou[:64-len(iou_samples)])
    else:
        LOGGER.info('Fork-based process pools unavailable or disabled; generating crops sequentially.')
        init_crop_worker()
        for name in tqdm(names,desc='Generate crops'):
            image_rows,image_iou=process_physical_image(name)
            rows.extend(image_rows)
            if len(iou_samples)<64:iou_samples.extend(image_iou[:64-len(iou_samples)])
    manifest=pd.DataFrame(rows)
    if manifest.empty or manifest.crop_id.duplicated().any():raise ValueError('Empty or duplicate crop manifest.')
    manifest.to_csv(YOLO_DATA_DIR/'crop_manifest.csv',index=False)
    (YOLO_DATA_DIR/'polygon_iou_sample.json').write_text(json.dumps(iou_samples,indent=2),encoding='utf-8')
    LOGGER.info('Crop generation complete | %s crops | %s strategies',len(manifest),manifest.strategy.value_counts().to_dict())
    return manifest,signature

def audit_crops(manifest):
    LOGGER.info('Starting crop audit for %s images and labels.',len(manifest))
    errors=[]
    for row in tqdm(manifest.itertuples(index=False),total=len(manifest),desc='Audit crop labels'):
        ip,lp=YOLO_DATA_DIR/row.image_path,YOLO_DATA_DIR/row.label_path
        if not ip.is_file() or not lp.is_file():errors.append(f'{row.crop_id}: missing path');continue
        im=cv2.imread(str(ip),cv2.IMREAD_GRAYSCALE)
        if im is None or im.shape!=(CROP_SIZE,CROP_SIZE):errors.append(f'{row.crop_id}: bad image size')
        text=lp.read_text(encoding='utf-8').strip()
        if row.strategy=='negative' and text:errors.append(f'{row.crop_id}: nonempty negative label')
        if row.strategy!='negative' and not text:errors.append(f'{row.crop_id}: empty positive label')
        for line_no,line in enumerate(text.splitlines(),1):
            try:
                p=line.split();xy=np.asarray([float(v) for v in p[1:]])
                valid=int(p[0])==0 and len(xy)>=6 and len(xy)%2==0 and np.isfinite(xy).all() and (xy>=0).all() and (xy<=1).all() and len(np.unique(xy.reshape(-1,2),axis=0))>=3
            except Exception:valid=False
            if not valid:errors.append(f'{row.crop_id}: bad polygon row {line_no}')
    if manifest.groupby('physical_filename').split.nunique().max()!=1:errors.append('split leakage')
    if not (manifest.crop_size==CROP_SIZE).all():errors.append('crop dimension mismatch')
    if manifest.loc[manifest.strategy=='negative','instance_rows'].ne(0).any():errors.append('negative has instances')
    negative=manifest.loc[manifest.strategy=='negative']
    if not negative.empty:
        # write_crop checked every negative against the complete safety mask before saving.
        # Rebuilding full-resolution masks here repeated expensive decoding after the audit bar.
        checked=negative['negative_safe_checked'].map(lambda value:str(value).strip().lower() in {'true','1'})
        if not checked.all():errors.append('negative safety provenance missing')
        LOGGER.debug('Verified safety-check provenance for %s negative crops.',len(negative))
    if errors:raise ValueError('Crop audit failed: '+'; '.join(errors[:20]))
    LOGGER.info('Crop audit complete | %s crops passed | %s negative crops carry safety-check provenance.',len(manifest),len(negative))
    return True

preprocess_start=time.perf_counter()
crop_manifest,source_preprocessing_signature=generate_crops()
audit_crops(crop_manifest)
preprocessing_seconds=time.perf_counter()-preprocess_start
LOGGER.info('Preprocessing complete | %.1f min | %s crops',preprocessing_seconds/60,len(crop_manifest))


## 9. Dataset visualization and preflight checks


In [ ]:
VISUAL_AUDIT_DIR=YOLO_DATA_DIR/'visual_audit'
VISUAL_AUDIT_DIR.mkdir(parents=True,exist_ok=True)
def show_views(name=None):
    if name is None:
        options=sorted(records.groupby('file_name').image_id.nunique().loc[lambda x:x>1].index)
        if not options:print('No multiply annotated image.');return
        name=options[0]
    image=cv2.imread(str(TRAIN_IMAGE_DIR/name),cv2.IMREAD_GRAYSCALE)
    group=records.loc[records.file_name==name]
    fig,axes=plt.subplots(1,len(group),figsize=(6*len(group),6),squeeze=False)
    for ax,r in zip(axes[0],group.itertuples(index=False)):
        ax.imshow(image,cmap='gray')
        for item in load_masks(str(r.image_id)):
            if item['mask'].any():ax.contour(item['mask'].astype(np.uint8),levels=[.5],linewidths=.7)
        ax.set_title(f'{r.annotation_view_id}: {r.image_id}');ax.axis('off')
    plt.tight_layout();fig.savefig(VISUAL_AUDIT_DIR/'two_annotation_views.png',dpi=120,bbox_inches='tight');plt.show()
def show_crops(manifest,n=2):
    for strategy in ['filament_centered','grid_context','negative']:
        group=manifest.loc[manifest.strategy==strategy].head(n)
        if group.empty:continue
        fig,axes=plt.subplots(1,len(group),figsize=(5*len(group),5),squeeze=False)
        for ax,r in zip(axes[0],group.itertuples(index=False)):
            im=cv2.imread(str(YOLO_DATA_DIR/r.image_path),cv2.IMREAD_GRAYSCALE);ax.imshow(im,cmap='gray')
            text=(YOLO_DATA_DIR/r.label_path).read_text(encoding='utf-8').strip()
            for line in text.splitlines() if text else []:
                xy=np.asarray([float(v) for v in line.split()[1:]]).reshape(-1,2)
                ax.plot(xy[:,0]*CROP_SIZE,xy[:,1]*CROP_SIZE,linewidth=.6)
            ax.set_title(strategy);ax.axis('off')
        plt.tight_layout();fig.savefig(VISUAL_AUDIT_DIR/f'{strategy}.png',dpi=120,bbox_inches='tight');plt.show()
show_views();show_crops(crop_manifest)
def display_crop_rows(rows,title):
    rows=rows.head(4)
    if rows.empty:
        print('No sample for',title);return
    fig,axes=plt.subplots(1,len(rows),figsize=(5*len(rows),5),squeeze=False)
    for ax,r in zip(axes[0],rows.itertuples(index=False)):
        im=cv2.imread(str(YOLO_DATA_DIR/r.image_path),cv2.IMREAD_GRAYSCALE);ax.imshow(im,cmap='gray')
        label=(YOLO_DATA_DIR/r.label_path).read_text(encoding='utf-8').strip()
        for line in label.splitlines() if label else []:
            xy=np.asarray([float(v) for v in line.split()[1:]]).reshape(-1,2)
            ax.plot(xy[:,0]*CROP_SIZE,xy[:,1]*CROP_SIZE,linewidth=.6)
        ax.set_title(f'{title} | {r.crop_x0},{r.crop_y0}');ax.axis('off')
    plt.tight_layout();fig.savefig(VISUAL_AUDIT_DIR/(title.replace(' ','_')+'.png'),dpi=120,bbox_inches='tight');plt.show()

# Show a crop with several labeled instances, crops near/outside an image boundary,
# and overlapping crops for the annotation requiring the most target windows.
multi=crop_manifest.loc[crop_manifest.instance_rows>=2]
display_crop_rows(multi,'multiple instances')
edge=crop_manifest.loc[(crop_manifest.crop_x0<0)|(crop_manifest.crop_y0<0)|
    (crop_manifest.crop_x1>2048)|(crop_manifest.crop_y1>2048)]
if edge.empty:
    edge=crop_manifest.loc[(crop_manifest.crop_x0<=0)|(crop_manifest.crop_y0<=0)|
        (crop_manifest.crop_x1>=2048)|(crop_manifest.crop_y1>=2048)]
display_crop_rows(edge,'image-edge crop')
target_rows=crop_manifest.loc[crop_manifest.target_annotation_ids!=''].copy()
if not target_rows.empty:
    target_rows['target_id']=target_rows.target_annotation_ids.str.split(';')
    exploded=target_rows.explode('target_id')
    repeated=exploded.groupby('target_id').filter(lambda g:len(g)>1)
    if not repeated.empty:
        largest=repeated.groupby('target_id').size().idxmax()
        display_crop_rows(repeated.loc[repeated.target_id==largest].sort_values(['crop_y0','crop_x0']),
                          'overlapping target crops')

strategy_counts=crop_manifest.strategy.value_counts().to_dict()
LOGGER.info('Crop strategies: %s',strategy_counts)
LOGGER.info('Crops by split: %s',crop_manifest.groupby('split').size().to_dict())
display(crop_manifest.groupby(['split','annotation_view_id']).size().groupby('split').describe().round(2))
display(crop_manifest.groupby(['split','physical_filename']).size().describe().round(2))
display(crop_manifest.loc[crop_manifest.visible_mask_area>0,'visible_mask_area'].describe(
    percentiles=[.25,.5,.75,.9,.99]).to_frame())
iou_sample=json.loads((YOLO_DATA_DIR/'polygon_iou_sample.json').read_text(encoding='utf-8'))
display(pd.DataFrame(iou_sample).mask_reconstruction_iou.describe().to_frame())
print('Polygon conversion notes: holes filled by external contours =',int(crop_manifest.polygon_holes.sum()),
      '| small components dropped =',int(crop_manifest.polygon_small_components_dropped.sum()),
      '| invalid components =',int(crop_manifest.polygon_invalid_components.sum()))
LOGGER.info('Preflight passed: paths, labels, dimensions, split grouping and negative safety.')
LOGGER.debug('Visual audit figures saved under %s',VISUAL_AUDIT_DIR)
shutil.copytree(VISUAL_AUDIT_DIR,ARTIFACT_DIR/'visual_audit',dirs_exist_ok=True)
# Display a crop where a labeled filament is only partially visible.
full_area={str(a['id']):float(a.get('area',0)) for a in raw_annotations}
def has_partial_instance(row):
    for value in str(row.visible_annotation_areas).split(';'):
        if ':' not in value:continue
        aid,visible_area=value.rsplit(':',1)
        if float(visible_area)<0.8*full_area.get(aid,float(visible_area)):
            return True
    return False
partial=crop_manifest.loc[crop_manifest.apply(has_partial_instance,axis=1)]
display_crop_rows(partial.sort_values('visible_mask_area'),'partially visible instance')


## 10. Finalize the preprocessing handoff bundle


In [ ]:
DATA_YAML=YOLO_DATA_DIR/'dataset.yaml'
dataset_yaml_text=f"path: '{YOLO_DATA_DIR.as_posix()}'\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n"
DATA_YAML.write_text(dataset_yaml_text,encoding='utf-8')
preprocessing_config={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_preprocessing_signature':source_preprocessing_signature,
 'source_dataset_root':str(DATA_ROOT),'annotation_path':str(ANNOTATION_PATH),
 'split_identifier':split_identifier,'physical_images':int(records.file_name.nunique()),
 'annotation_views':int(len(records)),'filament_instances':int(len(raw_annotations)),
 'crop_count':int(len(crop_manifest)),'crop_size':CROP_SIZE,
 'cpu_crop_workers':CPU_CROP_WORKERS,'cpu_affinity_count':CPU_AFFINITY_COUNT,
 'worker_native_threads':1,'training_performed':False,
 'strategy_counts':strategy_counts,
 'polygon_holes_filled':int(crop_manifest.polygon_holes.sum()),
 'polygon_small_components_dropped':int(crop_manifest.polygon_small_components_dropped.sum()),
 'polygon_invalid_components':int(crop_manifest.polygon_invalid_components.sum())}
audit={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_preprocessing_signature':source_preprocessing_signature,
 'source_dataset_root':str(DATA_ROOT),'annotation_path':str(ANNOTATION_PATH),
 'physical_images':int(records.file_name.nunique()),'annotation_views':int(len(records)),
 'filament_instances':int(len(raw_annotations)),'split_identifier':split_identifier,
 'crop_count':int(len(crop_manifest)),'strategy_counts':strategy_counts,
 'polygon_holes_filled':int(crop_manifest.polygon_holes.sum()),
 'polygon_small_components_dropped':int(crop_manifest.polygon_small_components_dropped.sum()),
 'polygon_invalid_components':int(crop_manifest.polygon_invalid_components.sum()),
 'crop_size':CROP_SIZE}
(YOLO_DATA_DIR/'dataset_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
(ARTIFACT_DIR/'dataset_audit.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
crop_config={
 'seed':SEED,'crop_size':CROP_SIZE,'grid_stride':GRID_STRIDE,
 'target_crop_stride':TARGET_CROP_STRIDE,'validation_fraction':VALIDATION_FRACTION,
 'minimum_visible_mask_area':MIN_VISIBLE_MASK_AREA,'negative_safety_margin':NEGATIVE_SAFETY_MARGIN,
 'exclude_any_annotated_filament_from_negatives':True,
 'negative_fraction_target':NEGATIVE_FRACTION_TARGET,
 'max_negative_crops_per_view':MAX_NEGATIVE_CROPS_PER_VIEW,
 'max_grid_crops_per_view':MAX_GRID_CROPS_PER_VIEW,
 'polygon_simplify_epsilon_pixels':POLYGON_SIMPLIFY_EPSILON}
(ARTIFACT_DIR/'preprocessing_config.json').write_text(json.dumps(preprocessing_config,indent=2),encoding='utf-8')
(ARTIFACT_DIR/'crop_config.json').write_text(json.dumps(crop_config,indent=2),encoding='utf-8')
mapping={'annotation_json':str(ANNOTATION_PATH),'views':[
 {'image_id':str(r.image_id),'physical_filename':r.file_name,'annotation_view_id':r.annotation_view_id,
  'annotation_ids':annotation_ids_by_image[str(r.image_id)]} for r in records.itertuples(index=False)]}
(ARTIFACT_DIR/'annotation_mapping.json').write_text(json.dumps(mapping,indent=2),encoding='utf-8')
for filename in ['crop_manifest.csv','dataset.yaml','split_manifest.csv','dataset_audit.json','polygon_iou_sample.json']:
    shutil.copy2(YOLO_DATA_DIR/filename,ARTIFACT_DIR/filename)
env={'python_version':platform.python_version(),'numpy_version':np.__version__,
 'pandas_version':pd.__version__,'opencv_version':cv2.__version__,
 'cpu_count':os.cpu_count(),'cpu_affinity_count':CPU_AFFINITY_COUNT,
 'crop_cpu_workers':CPU_CROP_WORKERS,'native_threads_per_crop_worker':1}
(ARTIFACT_DIR/'environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')
LOGGER.info('Prepared dataset: %s',YOLO_DATA_DIR)
LOGGER.info('Handoff bundle: %s',ARTIFACT_DIR)
LOGGER.debug('Handoff bundle includes prepared images/labels, manifests, split, mapping and audits.')


## Stage handoff

Save the output folder yolo26_crop_training_artifacts as a Kaggle dataset or attach it as notebook input to the GPU training notebook. It contains prepared_dataset with all crop images and labels, plus the crop manifest, split manifest, dataset YAML, annotation mapping, and preprocessing audit.

Run yolo26-crop-training-pipeline.ipynb in a separate Kaggle session configured for GPU T4 x2. Run yolo-post-processing.ipynb later in a CPU-only session.
